# Secondary-outcome enrichment

Observed 5-year incidence by predicted risk for the T2D, CKD, and MASLD complication models. People are ranked by `pred_risk_5yr` from the saved SOC and SOC+NMR absolute risks. The percentile figures show all 100 groups. The dumbbells show the top 1%, 5%, and 10%, with SOC as the reference.


In [ ]:
suppressPackageStartupMessages({
  library(tidyverse)
  library(here)
  library(survival)
})


In [ ]:
source(here("helpers/oof_enrichment.r"))


## Outcome configuration

Percentile y-axis maxima are set per outcome (`y_max`). Each outcome is read from the absolute-risk folder written by its model notebook.


In [ ]:
diabetes_risk_dir <- here("outputs/secondary_diabetes/absolute_risks")
kidney_risk_dir <- here("outputs/secondary_kidney/absolute_risks")
liver_risk_dir <- here("outputs/secondary_liver/absolute_risks")

table_dir <- here("outputs/tables")
figure_dir <- here("outputs/figures")
dir.create(table_dir, recursive = TRUE, showWarnings = FALSE)
dir.create(figure_dir, recursive = TRUE, showWarnings = FALSE)

t_star <- 5
risk_col <- sprintf("pred_risk_%dyr", t_star)
n_percentiles <- 100
top_props <- c(0.01, 0.05, 0.10)
family_order <- c("T2D", "CKD", "MASLD")
dumbbell_x_max <- c("T2D" = 90, "CKD" = 80, "MASLD" = 100)

model_palette <- c("SOC" = "#E64B35", "SOC+NMR" = "#4DBBD5")
legend_labels <- c("SOC" = "SOC(+)", "SOC+NMR" = "SOC(+) w/ NMR")

make_spec <- function(family, title, slug, risk_dir, y_max) {
  list(
    family = family,
    title = title,
    slug = slug,
    risk_dir = risk_dir,
    y_max = y_max,
    soc_file = paste0(slug, "_marker_fold_baseline_risk.rds"),
    nmr_file = paste0(slug, "_marker_nmr_fold_baseline_risk.rds")
  )
}

outcome_specs <- list(
  t2d_ckd = make_spec("T2D", "T2D to CKD", "t2d_ckd", diabetes_risk_dir, 0.90),
  t2d_nephropathy = make_spec("T2D", "T2D to Broad Renal Disease", "t2d_nephropathy", diabetes_risk_dir, 0.90),
  t2d_heart_failure = make_spec("T2D", "T2D to Heart Failure", "t2d_heart_failure", diabetes_risk_dir, 0.50),
  t2d_neuropathy = make_spec("T2D", "T2D to Neuropathy", "t2d_neuropathy", diabetes_risk_dir, 0.45),
  t2d_foot = make_spec("T2D", "T2D to Foot Pathology", "t2d_foot", diabetes_risk_dir, 0.40),
  t2d_retinopathy = make_spec("T2D", "T2D to Retinopathy", "t2d_retinopathy", diabetes_risk_dir, 0.50),

  ckd_heart_failure = make_spec("CKD", "CKD to Heart Failure", "ckd_heart_failure", kidney_risk_dir, 0.65),
  ckd_chd = make_spec("CKD", "CKD to Coronary Heart Disease", "ckd_chd", kidney_risk_dir, 0.40),
  ckd_afib = make_spec("CKD", "CKD to Atrial Fibrillation", "ckd_afib", kidney_risk_dir, 0.55),
  ckd_esrd = make_spec("CKD", "CKD to Kidney Failure", "ckd_esrd", kidney_risk_dir, 0.70),
  ckd_acs = make_spec("CKD", "CKD to Acute MI/ACS", "ckd_acs", kidney_risk_dir, 0.65),
  ckd_tia_stroke = make_spec("CKD", "CKD to TIA/Stroke", "ckd_tia_stroke", kidney_risk_dir, 0.55),

  nafld_diab = make_spec("MASLD", "MASLD to Type 2 Diabetes", "nafld_diab", liver_risk_dir, 0.80),
  nafld_ckd = make_spec("MASLD", "MASLD to CKD", "nafld_ckd", liver_risk_dir, 0.95),
  nafld_portalht = make_spec("MASLD", "MASLD to Portal Hypertension", "nafld_portalht", liver_risk_dir, 0.35),
  nafld_venous_thrombosis = make_spec("MASLD", "MASLD to Venous Thrombosis", "nafld_venous_thrombosis", liver_risk_dir, 0.25)
)

outcome_lookup <- bind_rows(lapply(outcome_specs, function(spec) {
  tibble(disease = spec$title, family = spec$family, slug = spec$slug, y_max = spec$y_max)
}))


## Load

SOC and SOC+NMR absolute risks are loaded once per outcome.


In [ ]:
load_model_pair <- function(spec) {
  soc_path <- file.path(spec$risk_dir, spec$soc_file)
  nmr_path <- file.path(spec$risk_dir, spec$nmr_file)
  missing <- c(soc_path, nmr_path)[!file.exists(c(soc_path, nmr_path))]
  if (length(missing) > 0) {
    stop("Missing absolute-risk RDS file(s):\n", paste(missing, collapse = "\n"))
  }
  list(SOC = readRDS(soc_path), `SOC+NMR` = readRDS(nmr_path))
}

oof_store <- lapply(outcome_specs, load_model_pair)
names(oof_store) <- names(outcome_specs)
cat("Loaded", length(oof_store), "secondary outcomes.\n")


## Observed incidence across risk percentiles

Each model is ranked into 100 groups. Observed incidence is the Kaplan–Meier estimate at 5 years.


In [ ]:
percentile_rows <- lapply(names(outcome_specs), function(id) {
  spec <- outcome_specs[[id]]
  bind_rows(lapply(names(oof_store[[id]]), function(model) {
    df <- compute_percentile_incidence_from_oof_risk(
      oof_store[[id]][[model]],
      t_star = t_star,
      risk_col = risk_col,
      n_percentiles = n_percentiles
    )
    df$family <- spec$family
    df$outcome <- spec$title
    df$slug <- spec$slug
    df$disease <- spec$title
    df$model <- model
    df$horizon_years <- t_star
    df
  }))
})

percentile_results <- bind_rows(percentile_rows) %>%
  mutate(
    family = factor(family, levels = family_order),
    outcome = factor(outcome, levels = outcome_lookup$disease),
    model = factor(model, levels = c("SOC", "SOC+NMR"))
  ) %>%
  arrange(family, outcome, model, risk_percentile)

attr(percentile_results, "t_star") <- t_star

write.csv(
  percentile_results,
  file.path(table_dir, "secondary_risk_percentile_groups_5yr.csv"),
  row.names = FALSE
)
for (family_name in family_order) {
  write.csv(
    percentile_results %>% filter(as.character(family) == family_name),
    file.path(table_dir, paste0("secondary_risk_percentile_groups_", tolower(family_name), "_5yr.csv")),
    row.names = FALSE
  )
}

percentile_figures <- setNames(lapply(family_order, function(family_name) {
  family_titles <- outcome_lookup$disease[outcome_lookup$family == family_name]
  y_limits <- setNames(
    as.list(outcome_lookup$y_max[match(family_titles, outcome_lookup$disease)]),
    family_titles
  )
  plot_percentile_incidence_faceted(
    percentile_results %>% filter(as.character(family) == family_name),
    y_limits = y_limits,
    t_star = t_star,
    ncol = if (family_name == "MASLD") 2 else 3,
    palette = model_palette
  ) +
    scale_color_manual(
      values = model_palette,
      labels = legend_labels,
      breaks = names(legend_labels)
    ) +
    labs(title = paste0(family_name, " secondary outcomes"))
}), family_order)

for (family_name in family_order) {
  dims <- if (family_name == "MASLD") c(width = 8, height = 7) else c(width = 11, height = 7)
  ggsave(
    file.path(figure_dir, paste0("secondary_risk_percentiles_", tolower(family_name), "_5yr.png")),
    percentile_figures[[family_name]],
    width = dims[["width"]],
    height = dims[["height"]],
    dpi = 600
  )
}


In [ ]:
options(repr.plot.width = 7, repr.plot.height = 6)
percentile_figures$T2D


In [ ]:
options(repr.plot.width = 7, repr.plot.height = 6)

percentile_figures$CKD


In [ ]:
options(repr.plot.width = 6, repr.plot.height = 6)

percentile_figures$MASLD


## Top-risk enrichment

Observed 5-year incidence, and raw and IPCW event capture, among people in the top 1%, 5%, and 10% of predicted risk. Deltas use SOC as the reference.


In [ ]:
disease_models <- setNames(
  lapply(names(outcome_specs), function(id) oof_store[[id]]),
  vapply(outcome_specs, function(spec) spec$title, character(1))
)

top_incidence <- top_risk_incidence_from_oof_risk(
  disease_models,
  t_star = t_star,
  risk_col = risk_col,
  top_props = top_props,
  reference_model = "SOC"
)

top_risk_clean <- top_incidence %>%
  mutate(disease = as.character(disease)) %>%
  left_join(outcome_lookup %>% select(disease, family, slug), by = "disease") %>%
  mutate(
    family = factor(family, levels = family_order),
    outcome = factor(disease, levels = outcome_lookup$disease),
    model = factor(as.character(model), levels = c("SOC", "SOC+NMR")),
    top_label = factor(as.character(top_label), levels = c("Top 1%", "Top 5%", "Top 10%"))
  ) %>%
  select(
    family, outcome, slug, model, top_label, t_star, top_prop,
    n_total, n_top, n_events_top_by_tstar,
    obs_incidence, obs_lower, obs_upper, delta_obs_incidence_vs_ref,
    event_capture_raw, delta_event_capture_raw_vs_ref,
    event_capture_ipcw, delta_event_capture_ipcw_vs_ref
  ) %>%
  arrange(family, outcome, top_label, model)

write.csv(
  top_risk_clean,
  file.path(table_dir, "secondary_top_risk_incidence_5yr.csv"),
  row.names = FALSE
)
for (family_name in family_order) {
  write.csv(
    top_risk_clean %>% filter(as.character(family) == family_name),
    file.path(table_dir, paste0("secondary_top_risk_incidence_", tolower(family_name), "_5yr.csv")),
    row.names = FALSE
  )
}

top_risk_clean


## Top-risk figures

One dumbbell per family. Points are observed 5-year incidence in each top-risk group.


In [ ]:
plot_family_dumbbell <- function(top_df, family_name) {
  family_titles <- outcome_lookup$disease[outcome_lookup$family == family_name]
  paired <- top_df %>%
    filter(as.character(family) == family_name) %>%
    mutate(obs_incidence_pct = 100 * obs_incidence) %>%
    select(outcome, top_label, model, obs_incidence_pct) %>%
    pivot_wider(names_from = model, values_from = obs_incidence_pct) %>%
    mutate(outcome = factor(as.character(outcome), levels = rev(family_titles)))

  x_max <- dumbbell_x_max[[family_name]]

  ggplot(paired, aes(y = outcome)) +
    geom_segment(
      aes(x = SOC, xend = `SOC+NMR`, yend = outcome),
      linewidth = 0.75,
      color = "grey68"
    ) +
    geom_point(aes(x = SOC, color = "SOC", shape = "SOC"), size = 2) +
    geom_point(aes(x = `SOC+NMR`, color = "SOC+NMR", shape = "SOC+NMR"), size = 2) +
    facet_wrap(~ top_label, nrow = 1) +
    scale_color_manual(
      values = model_palette,
      labels = legend_labels,
      breaks = names(legend_labels)
    ) +
    scale_shape_manual(
      values = c("SOC" = 16, "SOC+NMR" = 17),
      labels = legend_labels,
      breaks = names(legend_labels)
    ) +
    scale_x_continuous(
      limits = c(0, x_max),
      breaks = seq(0, x_max, by = 20),
      labels = function(x) paste0(x, "%"),
      expand = expansion(mult = c(0.02, 0.05))
    ) +
    labs(
      title = paste0(family_name, " top-risk observed incidence"),
      x = "Observed 5-year incidence",
      y = NULL,
      color = NULL,
      shape = NULL
    ) +
    theme_bw(base_size = 10) +
    theme(
      plot.title = element_text(size = 13, face = "bold", hjust = 0.5),
      strip.background = element_blank(),
      strip.text = element_text(face = "bold"),
      panel.grid.minor = element_blank(),
      legend.position = "bottom",
      panel.spacing.x = grid::unit(1.15, "lines")
    )
}

dumbbell_figures <- setNames(
  lapply(family_order, function(family_name) plot_family_dumbbell(top_risk_clean, family_name)),
  family_order
)

for (family_name in family_order) {
  dims <- if (family_name == "MASLD") c(width = 6.5, height = 2.15) else c(width = 6.5, height = 2.3)
  ggsave(
    file.path(figure_dir, paste0("secondary_top_risk_incidence_", tolower(family_name), "_5yr.png")),
    dumbbell_figures[[family_name]],
    width = dims[["width"]],
    height = dims[["height"]],
    dpi = 600
  )
}


In [ ]:
options(repr.plot.width = 6, repr.plot.height = 2.5)
dumbbell_figures$T2D


In [ ]:
options(repr.plot.width = 6, repr.plot.height = 2.5)
dumbbell_figures$CKD


In [ ]:
options(repr.plot.width = 6, repr.plot.height = 2.25)
dumbbell_figures$MASLD
